In [0]:
# ==============================================================================
# MOVER ARQUIVOS PROCESSADOS PARA HISTORY/
# ==============================================================================
# Move os arquivos Excel processados de current/ para history/ após o
# sucesso do SCD2. Valida que current/ ficou vazio ao final.
# Em caso de falha na movimentação, levanta RuntimeError para evitar
# reprocessamento na próxima execução.
# ==============================================================================

# ==============================================================================
# MOVER TODOS OS ARQUIVOS DE current/ PARA history/
# ==============================================================================

SOURCE_PATH = "/Volumes/parts_hdbk_sandbox/pr_cadastrao/sap_cadastraorefinado/current/"
HISTORY_PATH = "/Volumes/parts_hdbk_sandbox/pr_cadastrao/sap_cadastraorefinado/history/"

moved = []
failed = []

# Lista todos os arquivos existentes em current/
files = dbutils.fs.ls(SOURCE_PATH)

if not files:
    print("Nenhum arquivo encontrado em current/")
else:

    for file in files:

        # Ignora diretórios caso existam
        if file.isDir():
            continue

        source = file.path
        destination = f"{HISTORY_PATH}{file.name}"

        try:
            dbutils.fs.mv(source, destination)
            moved.append(file.name)
            print(f"Movido: {file.name} → history/")

        except Exception as e:
            failed.append((file.name, str(e)))
            print(f"ERRO ao mover {file.name}: {e}")

print(f"\nResumo: {len(moved)} arquivo(s) movido(s), {len(failed)} erro(s)")

if failed:
    raise RuntimeError(
        f"Falha ao mover {len(failed)} arquivo(s): "
        + "; ".join(f"{name}: {err}" for name, err in failed)
    )

# ------------------------------------------------------------------------------
# Validação final
# ------------------------------------------------------------------------------

remaining = [
    f for f in dbutils.fs.ls(SOURCE_PATH)
    if not f.isDir()
]

if remaining:
    raise RuntimeError(
        f"Restaram {len(remaining)} arquivo(s) em current/: "
        + ", ".join(f.name for f in remaining)
    )

print("✓ Diretório current/ vazio após movimentação.")